In [ ]:
import sys
sys.path.append("/home/arisrei/research/RPL/ORION/third_party/XMem")
dust3r_path = './third_party/dust3r'
sys.path.append(dust3r_path)

print(sys.path)

In [ ]:
import numpy as np
import open3d as o3d
import plotly.graph_objects as go
import os
import PIL
from PIL import Image
from tqdm import tqdm

#add parent path to sys.path to import from orion
from orion.utils.o3d_utils import *
from orion.utils.misc_utils import *
from orion.algos.xmem_tracker import XMemTracker
from dust3r.utils.geometry import geotrf
import numpy as np


In [ ]:
def dust3r_resize(img, size, square_ok=False):
    if size == 224:
        # resize short side to 224 (then crop)
        img = _resize_pil_image(img, round(size * max(W1/H1, H1/W1)))
    else:
        # resize long side to 512
        img = _resize_pil_image(img, size)
    W, H = img.size
    cx, cy = W//2, H//2
    if size == 224:
        half = min(cx, cy)
        img = img.crop((cx-half, cy-half, cx+half, cy+half))
    else:
        halfw, halfh = ((2*cx)//16)*8, ((2*cy)//16)*8
        if not (square_ok) and W == H:
            halfh = 3*halfw/4
        img = img.crop((cx-halfw, cy-halfh, cx+halfw, cy+halfh))
    return img

def _resize_pil_image(img, long_edge_size):
    S = max(img.size)
    if S > long_edge_size:
        interp = PIL.Image.LANCZOS
    elif S <= long_edge_size:
        interp = PIL.Image.BICUBIC
    new_size = tuple(int(round(x*long_edge_size/S)) for x in img.size)
    return img.resize(new_size, interp)


In [ ]:
def create_point_clouds_from_keypoints(keypoints, depth_img, camera_intrinsics_matrix):
    """Manual computation of backprojection. This is used to compute every single point clouds based on a list of given keypoints.

    Args:
        keypoints (np.ndarray): (N, 2) keypoints in (x,y)
        depth_img (np.ndarray): HxW depth image
        camera_intrinsics_matrix (np.ndarray): 3x3 camera intrinsics matrix

    Returns:
        _type_: _description_
    """
    z = depth_img.squeeze() / 1000
    
    if len(depth_img.shape) == 3:
        height, width = depth_img[0].shape[:2]
    else:
        height, width = depth_img.shape[:2]
    x = keypoints[:, 0]
    y = keypoints[:, 1]

    print("depth y and x", z.shape, np.max(y), np.max(x))

    new_z = z[y.astype(np.int32), x.astype(np.int32)]
    fx = camera_intrinsics_matrix[0, 0]
    fy = camera_intrinsics_matrix[1, 1]
    cx = camera_intrinsics_matrix[0, 2]
    cy = camera_intrinsics_matrix[1, 2]

    X = (x - cx) * new_z / fx
    Y = (y - cy) * new_z / fy
    Z = new_z

    points = np.ascontiguousarray(np.stack([X, Y, Z], axis=-1))
    #print(points.shape)
    #points = points.reshape(-1, 3)
    #print(points.shape, "new")
    return points


def visualize_o3d_point_cloud(o3d_pcd):
    point_cloud = np.array(o3d_pcd.points)
    colors_rgb = np.array(o3d_pcd.colors)

    color_str = ['rgb('+str(r)+','+str(g)+','+str(b)+')' for r,g,b in colors_rgb]

    # Extract x, y, and z columns from the point cloud
    x_vals = point_cloud[:, 0]
    y_vals = point_cloud[:, 1]
    z_vals = point_cloud[:, 2]

    # Create the scatter3d plot
    rgbd_scatter = go.Scatter3d(
        x=x_vals,
        y=y_vals,
        z=z_vals,
        mode='markers',
        marker=dict(size=3, color=color_str, opacity=0.8)
    )

    # Set the layout for the plot
    layout = go.Layout(
        margin=dict(l=0, r=0, b=0, t=0)
    )

    fig = go.Figure(data=[rgbd_scatter], layout=layout)
    # Show the figure
    return fig

In [ ]:
annotation_dir = "/home/arisrei/research/RPL/ORION/annotations/human_demo/yellow_bottle"
dust3r_dir = os.path.join(annotation_dir, "dust3r_demo_gui")
rgb_imgs = np.load(os.path.join(dust3r_dir, "images.npy"))
depths = np.load(os.path.join(dust3r_dir, "depths.npy"))
intrinsics = np.load(os.path.join(dust3r_dir, "intrinsics.npy"))
extrinsics = np.load(os.path.join(dust3r_dir, "extrinsics.npy"))
img_indexes = np.array([0, 30, 60, 90, 126])
mask_orig = np.load(os.path.join(annotation_dir, "masks.npz"))
mask_orig = mask_orig['arr_0']
mask = np.zeros((rgb_imgs.shape[0:3]), dtype=np.uint8)

images_unmasked = None
if os.path.isfile(os.path.join(annotation_dir, "images_unmasked.npy")):
    images_unmasked = np.load(os.path.join(annotation_dir, "images_unmasked.npy"))

#

In [ ]:
config = json.load(open(os.path.join(annotation_dir, "config.json")))
demo_file = h5py.File(config['original_file'], "r+")

dust3r_rgb = demo_file["data/human_demo/obs/dust3r_rgb"][()]
agentview_rgb = demo_file["data/human_demo/obs/agentview_rgb"][()]

resized_dust3r = np.array(dust3r_resize(Image.fromarray(agentview_rgb[0]), 512))
print(resized_dust3r.shape)

dust3r_mask = np.zeros((agentview_rgb.shape[0], resized_dust3r.shape[0], resized_dust3r.shape[1]), dtype=np.uint8)
for i in range(agentview_rgb.shape[0]):
    dust3r_mask[i] = np.array(Image.fromarray(mask_orig[i]).resize((dust3r_rgb.shape[2], dust3r_rgb.shape[1]), resample=PIL.Image.NEAREST))

demo_file.close()

In [ ]:
display(Image.fromarray(dust3r_mask[0] * 100))
tmp = np.where(dust3r_mask[0] == 1, 1, 0).astype(np.uint8)
display(Image.fromarray(tmp * 100))

In [ ]:
target_ob_mask = 1
mask_orig[mask_orig != target_ob_mask] = 0
for i in range(img_indexes.shape[0]):
    dust3r_resized = dust3r_resize(Image.fromarray(mask_orig[img_indexes[i]]), 512)
    mask[i] = np.array(dust3r_resized)    
masked = rgb_imgs[i] * 255 * mask[i][:,:,np.newaxis]
display(Image.fromarray((rgb_imgs[i] * 255).astype(np.uint8)))
display(Image.fromarray(masked.astype(np.uint8)))    
display(Image.fromarray(mask[0] * 255))

In [ ]:
all_object_pts = []
all_object_colors = []
#all_boject_
for q in tqdm(range(img_indexes.shape[0])):
    keypoints = []
    colors = []
    trunc = 1000

    #depths[q][mask[q] != 1] = trunc * 1000000

    print(rgb_imgs[q].shape)

    for i in range(rgb_imgs[q].shape[0]):
        for j in range(rgb_imgs[q].shape[1]):
            if mask[q][i, j] == 1:
                keypoints.append(np.asarray([j, i]))
                if type(images_unmasked) == np.ndarray:
                    colors.append(images_unmasked[q][i, j])
                else:
                    colors.append(rgb_imgs[q][i, j])

    keypoints = np.asarray(keypoints)
    colors = np.asarray(colors)
    points_3d = create_point_clouds_from_keypoints(keypoints, depths[q], intrinsics[q])
    #points_3d = create_point_clouds_from_keypoints(keypoints, depths[q],
    #intrinsics[q])
    points_3d *= 1000
    points_3d = geotrf(extrinsics[q], points_3d)

    #visualize in open3d
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points_3d)
    pcd.colors = o3d.utility.Vector3dVector(colors)
    print(np.array(pcd.points[0:5]))
    #pcd.transform(extrinsics[q])
    all_object_pts.append(np.array(pcd.points))
    all_object_colors.append(np.array(pcd.colors))


    #o3d.io.write_point_cloud(os.path.join(dust3r_dir, f"image_{img_indexes[q]}_pcd_unmasked_input.ply"), pcd)

    fig = visualize_o3d_point_cloud(pcd)
    fig.write_html(os.path.join(dust3r_dir, f"image_{img_indexes[q]}_pcd_unmasked_input.html"))
    print("saving to ", os.path.join(dust3r_dir, f"image_{img_indexes[q]}_pcd_unmasked_input.html"))
    display(fig)


In [ ]:
print(img_indexes)
print(extrinsics.shape)
for i in range(img_indexes.shape[0]):
    print(extrinsics[i])

In [ ]:
np.save(os.path.join(dust3r_dir, "all_obj_pts.npy"), np.array(all_object_pts, dtype=object))
np.save(os.path.join(dust3r_dir, "all_obj_colors.npy"), np.array(all_object_colors, dtype=object))